In [ ]:
import pandas as pd

# Cargar el dataset
df = pd.read_csv('../data/EMOEVAL_ES/emoevent_es.csv', sep='\t')

df.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8409 entries, 0 to 8408
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         8409 non-null   int64 
 1   tweet      8409 non-null   object
 2   emotion    8409 non-null   object
 3   offensive  8409 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 262.9+ KB


In [2]:
import pandas as pd
import dtale

# Mostrar el DataFrame en D-Tale
# d = dtale.show(df)
# d.open_browser() 
# d._main_url
# dtale.utils.get_open_dtales() 
 

In [3]:
df = df.drop(columns=["id"] )

In [4]:
# df

In [5]:
# Contar las emociones ofensivas
offensive_emotion_counts = df[df['offensive'] == 1]['emotion'].value_counts()

# Convertir a DataFrame para visualización
offensive_emotion_counts_df = offensive_emotion_counts.reset_index()
offensive_emotion_counts_df.columns = ['emotion', 'offensive_count']

In [6]:
df['emotion'].value_counts()

emotion
others      4127
joy         1815
sadness     1009
anger        857
surprise     344
disgust      161
fear          96
Name: count, dtype: int64

In [7]:
offensive_emotion_counts_df

,emotion,offensive_count
0,anger,378
1,others,111
2,disgust,94
3,joy,72
4,surprise,37
5,sadness,10
6,fear,4


In [8]:
df = df.drop(columns=["offensive"] )

In [9]:
df.rename(columns={'tweet': 'text'}, inplace=True)

In [10]:
filas_repetidas = df[df.duplicated()]
filas_repetidas

,text,emotion


In [11]:
emotion_distribution = df['emotion'].value_counts()

# Mostrar la distribución
print(emotion_distribution)

emotion
others      4127
joy         1815
sadness     1009
anger        857
surprise     344
disgust      161
fear          96
Name: count, dtype: int64


In [12]:
# Eliminar las filas donde emotion sea 'surprise'
df = df[df["emotion"] != "surprise"].copy()

# Verificamos que ya no esté 'surprise' en la distribución
print(df["emotion"].value_counts())


emotion
others     4127
joy        1815
sadness    1009
anger       857
disgust     161
fear         96
Name: count, dtype: int64


In [13]:
# df

In [14]:
# Mapeo de etiquetas de inglés a español
emotion_mapping = {
    'others': 'neutral',#segun indica la página oficial https://competitions.codalab.org/competitions/28682
    'anger': 'ira',
    'disgust': 'disgusto',
    'sadness': 'tristeza',
    'joy': 'alegría',
    'fear': 'miedo',
    # 'surprise': 'sorpresa'
}

df['emotion'] = df['emotion'].map(emotion_mapping) # columna 'emotion' con valores en español

In [15]:
print(df['emotion'].value_counts())

emotion
neutral     4127
alegría     1815
tristeza    1009
ira          857
disgusto     161
miedo         96
Name: count, dtype: int64


In [16]:
print(df['emotion'].value_counts())

emotion
neutral     4127
alegría     1815
tristeza    1009
ira          857
disgusto     161
miedo         96
Name: count, dtype: int64


In [17]:
# df

In [18]:
import re

def get_words_around_hashtag(text):
    # Patrón para encontrar palabra antes y después de HASHTAG
    match = re.search(r'(\w+)?\s*HASHTAG\s*(\w+)?', text)
    if match:
        return match.group(1), match.group(2)
    else:
        return None, None

# Aplicamos la función a cada tweet
df[['word_before', 'word_after']] = df['text'].apply(lambda x: pd.Series(get_words_around_hashtag(str(x))))
# Filtrar los que tienen HASHTAG
df_hashtag = df[df['text'].str.contains("HASHTAG", na=False)]

# Mostrar los resultados
df_hashtag[['text', 'word_before', 'word_after']]

,text,word_before,word_after
0,Acabo de ver la gran pérdida que estamos tenie...,en,None
1,USER ¿A que vamos a reconstruir Notre Dame ant...,None,URL
2,Desde ayer andan sufriendo por el incendio de ...,de,y
3,Muy afectada -como muchísima gente- por lo ocu...,a,None
4,Es una mierda lo que paso en HASHTAG pero plea...,en,pero
...,...,...,...
8404,"Dembele es un imbécil, neta es increíble lo pe...",None,HASHTAG
8405,HASHTAG Puta q son desagradables los wnes del ...,None,Puta
8406,Puta que me cae mal Suárez ctm HASHTAG,ctm,None
8407,"Como te odio USER, odio tus comentarios filosó...",None,HASHTAG


In [19]:
# Contar y filtrar palabras antes de HASHTAG con frecuencia > 1
before_common = df['word_before'].value_counts()
before_common = before_common[before_common > 1]


In [20]:
# Contar y filtrar palabras después de HASHTAG con frecuencia > 1
after_common = df['word_after'].value_counts()
after_common = after_common[after_common > 2]
print(after_common)

word_after
HASHTAG       2291
URL            561
y              223
en             120
es             117
              ... 
qué              3
cada             3
eres             3
creo             3
celebramos       3
Name: count, Length: 121, dtype: int64


In [21]:
# Convertir a DataFrame si quieres trabajar con ellas
before_df = before_common.reset_index()
before_df.columns = ['word_before', 'frequency']

after_df = after_common.reset_index()
after_df.columns = ['word_after', 'frequency']


In [22]:
import re

def clean_tweet(tweet):
    tweet = re.sub(r'  ', ' ', tweet)
    tweet = re.sub(r'@USER', ' ', tweet)
    tweet = re.sub(r'USER', ' ', tweet)
    tweet = re.sub(r'DE HASHTAG', ' ', tweet)
    tweet = re.sub(r'En HASHTAG', ' ', tweet)
    tweet = re.sub(r'El HASHTAG', ' ', tweet)
    tweet = re.sub(r'La HASHTAG', ' ', tweet)
    tweet = re.sub(r' del HASHTAG', ' ', tweet)
    tweet = re.sub(r' de HASHTAG', ' ', tweet)
    tweet = re.sub(r' por HASHTAG', ' ', tweet)
    tweet = re.sub(r' en HASHTAG', ' ', tweet)
    tweet = re.sub(r' el HASHTAG', ' ', tweet)
    tweet = re.sub(r' las HASHTAG', ' ', tweet)
    tweet = re.sub(r' a los HASHTAG', ' ', tweet)
    tweet = re.sub(r' los HASHTAG', ' ', tweet)
    tweet = re.sub(r' al HASHTAG', ' ', tweet)
    tweet = re.sub(r' la HASHTAG', ' ', tweet)
    tweet = re.sub(r' la HASHTAG', ' ', tweet)
    tweet = re.sub(r' ni HASHTAG', ' ', tweet)
    tweet = re.sub(r' e HASHTAG', ' ', tweet)
    tweet = re.sub(r' a HASHTAG', ' ', tweet)
    tweet = re.sub(r' o HASHTAG', ' ', tweet)
    tweet = re.sub(r' es HASHTAG', ' ', tweet)
    tweet = re.sub(r' con HASHTAG', ' ', tweet)
    tweet = re.sub(r' como HASHTAG', ' ', tweet)
    tweet = re.sub(r' para HASHTAG', ' ', tweet)
    tweet = re.sub(r' y HASHTAG', ' ', tweet)
    tweet = re.sub(r' Este HASHTAG,', ' ', tweet)
    tweet = re.sub(r'HASHTAG', ' ', tweet)
    tweet = re.sub(r' y URL', ' ', tweet)
    tweet = re.sub(r'URL', ' ', tweet)
    tweet = re.sub(r'l@s', 'los', tweet)
    tweet = re.sub(r'L@s', 'los', tweet)
    tweet = re.sub(r'tod@s', 'todos', tweet)
    tweet = re.sub(r'Tod@s', 'todos', tweet)
    tweet = re.sub(r'est@s', 'estos', tweet)
    tweet = re.sub(r'alumn@s', 'alumnos', tweet)
    tweet = re.sub(r'hij@s', 'hijos', tweet)
    tweet = re.sub(r'niñ@s', 'niños', tweet)
    tweet = re.sub(r'nosotr@s', 'nosotros', tweet)
    tweet = re.sub(r'compañer@s', 'compañeros', tweet)
    tweet = re.sub(r'católic@s', 'católicos', tweet)
    tweet = re.sub(r'http\S+|www.\S+', ' ', tweet)  # eliminar urls
    # tweet = re.sub(r'[^a-zA-ZñÑáéíóúÁÉÍÓÚ\s]', '', tweet)  # eliminar símbolos
    # tweet = tweet.lower()
    tweet = re.sub(r'\s+', ' ', tweet).strip()  # eliminar espacios extras
    return tweet

df['text'] = df['text'].apply(clean_tweet)


In [23]:
from collections import Counter

# Unir todos los textos en uno solo
all_text = ' '.join(df['text'].astype(str))

# Contar cada caracter
char_counts = Counter(all_text)

# Mostrar los caracteres y sus conteos ordenados por frecuencia
# for char, count in char_counts.most_common():
#     print(f"'{char}': {count}")

In [24]:
import re

# Paso 1: eliminar texto entre corchetes
df['text'] = df['text'].astype(str).str.replace(r'\[.*?\]', '', regex=True)

# Paso 2: listar palabras que contienen @
palabras_con_arroba = df['text'].str.findall(r'\S*@\S+')

# Paso 3: eliminar palabras que contienen @
df['text'] = df['text'].str.replace(r'\S*@\S+', '', regex=True)

# Paso 4: corregir acentos mal escritos
df['text'] = (
    df['text']
    .str.replace('à', 'á')
    .str.replace('è', 'é')
    .str.replace('ì', 'í')
    .str.replace('ò', 'ó')
    .str.replace('ù', 'ú')
    .str.replace('À', 'Á')
    .str.replace('È', 'É')
    .str.replace('Ì', 'Í')
    .str.replace('Ò', 'Ó')
    .str.replace('Ù', 'Ú')
)

In [25]:
import pandas as pd
import dtale

# Mostrar el DataFrame en D-Tale
d = dtale.show(palabras_con_arroba)
d.open_browser() 


In [26]:
def limpiar_texto(texto):
    # Reemplaza caracteres no permitidos por espacio
    return ''.join(c if re.match(caracteres_permitidos, c) else ' ' for c in texto)

# Letras latinas con tildes y ñ
letras = 'a-zA-ZáéíóúÁÉÍÓÚñÑüÜçÇ'

# Dígitos
digitos = '0-9'

# Signos de puntuación básicos
puntuacion = r'\.\,\!\¡\?\¿\;'

# Emojis de emociones: puedes ampliarlos si usas más
emojis = '❤️😂😭😱😢😍😔🤔🙏💪😜😎😊😡🤬🤗🤩😅😉😘🥰😇🙂🙃🥺😖😤😨😩😰😓😥😿😾😼🙀🙈🙉🙊'

# Todos juntos
caracteres_permitidos = f"[{letras}{puntuacion}{emojis} ]" #{digitos}

df['text_limpio'] = df['text'].astype(str).apply(limpiar_texto)

from collections import Counter

contador = Counter(''.join(df['text_limpio']))
for char, count in contador.most_common():
    print(f"'{char}': {count}")

' ': 156960
'e': 83905
'a': 75521
'o': 59076
's': 47587
'n': 43077
'r': 40986
'i': 36949
'l': 35572
'd': 30443
't': 27693
'u': 26774
'c': 23061
'm': 18537
'p': 15787
'.': 10136
'b': 7488
'g': 7339
'q': 7269
'v': 7079
'h': 6715
'y': 6680
',': 5574
'f': 4159
'E': 4112
'í': 3509
'á': 3285
'ó': 3168
'A': 3045
'j': 2922
'L': 2596
'!': 2462
'z': 2461
'S': 2286
'C': 2070
'M': 2041
'N': 1987
'O': 1957
'é': 1935
'P': 1817
'D': 1588
'ñ': 1539
'T': 1351
'R': 1314
'I': 1198
'?': 1046
'U': 1009
'V': 916
'H': 848
'B': 819
'x': 772
'Q': 768
'ú': 735
'G': 735
'F': 662
'Y': 641
'¡': 532
'¿': 507
'️': 499
'J': 403
'k': 168
'❤': 152
'😂': 132
'w': 104
'ç': 96
'😭': 90
';': 89
'X': 87
'Z': 86
'😢': 76
'😍': 75
'É': 74
'Ó': 73
'🙏': 68
'Í': 68
'💪': 65
'🤔': 63
'😱': 59
'Ñ': 57
'Á': 53
'ü': 52
'K': 45
'W': 40
'😔': 37
'😉': 31
'😜': 29
'😎': 26
'😊': 25
'Ú': 20
'🤩': 17
'😥': 14
'🤗': 14
'😓': 11
'😘': 11
'😅': 11
'🥰': 10
'😰': 9
'Ç': 9
'🙈': 8
'😖': 8
'🥺': 7
'😨': 6
'😡': 5
'🤬': 5
'😩': 5
'🙊': 4
'🙂': 3
'🙀': 2
'😤': 2
'🙉': 2
'Ü': 2

In [27]:
# gui3 = show(df)

In [28]:
# df

In [29]:
df['text'] = df['text_limpio']
df = df.drop(columns=["word_before", "word_after", "text_limpio"] )

In [30]:
import re
# Función para limpiar texto
def limpiar_texto(texto):
    texto = str(texto)  # Por si hay NaNs u objetos raros
    texto = texto.strip()  # Eliminar espacios al principio y al final
    texto = re.sub(r'\s+', ' ', texto)  # Reemplaza múltiples espacios por uno
    return texto

# Limpiar textos
df["text"] = df["text"].apply(limpiar_texto)

In [31]:
# Eliminar puntos y comas al inicio del texto
df['text'] = df['text'].str.replace(r'^[.,]+', '', regex=True).str.lstrip()

In [32]:
# !pip install fasttext-wheel

In [ ]:
import fasttext
import pandas as pd
from tqdm.notebook import tqdm
import re

# Cargar modelo fastText entrenado para identificación de idiomas
model_path = r"C:\Users\leo23\OneDrive\Documentos\Fine_tuning_llama2\models\fasttext\models--facebook--fasttext-language-identification\snapshots\3af127d4124fc58b75666f3594bb5143b9757e78\model.bin"
language_detector = fasttext.load_model(model_path)

# Función mejorada para preprocesar y detectar idioma
def detect_language(text: str, threshold: float = 0.75) -> str:
    try:
        if pd.isna(text) or len(text.strip()) < 3:
            return "unknown"
        
        # Limpieza del texto
        clean_text = re.sub(r'[^\w\s]', '', text).strip().replace("\n", " ")
        
        # Obtener las 3 predicciones más probables
        labels, probabilities = language_detector.predict(clean_text, k=3)
        
        if probabilities[0] < threshold:
            return "low_confidence"
        
        return labels[0].split("__")[-1]
    except Exception as e:
        return "error"

# Aplicar detección con progreso
tqdm.pandas(desc="Detectando idiomas (sensitivo)")
df["idioma"] = df["text"].progress_apply(detect_language)

# Guardar resultados
df.to_csv("../data/EMOEVAL_ES/Emoevent_dataset_con_idioma.csv", index=False)

# Mostrar ejemplo
print("Distribución de idiomas detectados:")
print(df["idioma"].value_counts())

df[["text", "idioma"]].head()

Detectando idiomas (sensitivo):   0%|          | 0/8065 [00:00<?, ?it/s]

Distribución de idiomas detectados:
idioma
spa_Latn          7579
low_confidence     330
yue_Hant            74
kor_Hang            35
glg_Latn             9
ast_Latn             8
por_Latn             8
ita_Latn             4
cat_Latn             4
epo_Latn             3
fra_Latn             2
lmo_Latn             1
hun_Latn             1
deu_Latn             1
tsn_Latn             1
gaz_Latn             1
ces_Latn             1
vec_Latn             1
pap_Latn             1
oci_Latn             1
Name: count, dtype: int64


,text,idioma
0,Acabo de ver la gran pérdida que estamos tenie...,spa_Latn
1,¿A que vamos a reconstruir Notre Dame antes de...,spa_Latn
2,Desde ayer andan sufriendo por el incendio y n...,spa_Latn
3,Muy afectada como muchísima gente por lo ocurr...,spa_Latn
4,Es una mierda lo que paso pero please dejen de...,spa_Latn


In [35]:
# Contar la cantidad de ocurrencias por cada idioma
conteo_idiomas = df["idioma"].value_counts()
print("Conteo de idiomas identificados:")
print(conteo_idiomas)

# Muestra los primeros registros del dataset con la columna "idioma"
display(df[["text", "idioma"]])

Conteo de idiomas identificados:
idioma
spa_Latn          7579
low_confidence     330
yue_Hant            74
kor_Hang            35
glg_Latn             9
ast_Latn             8
por_Latn             8
ita_Latn             4
cat_Latn             4
epo_Latn             3
fra_Latn             2
lmo_Latn             1
hun_Latn             1
deu_Latn             1
tsn_Latn             1
gaz_Latn             1
ces_Latn             1
vec_Latn             1
pap_Latn             1
oci_Latn             1
Name: count, dtype: int64


,text,idioma
0,Acabo de ver la gran pérdida que estamos tenie...,spa_Latn
1,¿A que vamos a reconstruir Notre Dame antes de...,spa_Latn
2,Desde ayer andan sufriendo por el incendio y n...,spa_Latn
3,Muy afectada como muchísima gente por lo ocurr...,spa_Latn
4,Es una mierda lo que paso pero please dejen de...,spa_Latn
...,...,...
8404,"Dembele es un imbécil, neta es increíble lo pe...",spa_Latn
8405,Puta q son desagradables los wnes del Barca en...,spa_Latn
8406,Puta que me cae mal Suárez ctm,low_confidence
8407,"Como te odio , odio tus comentarios filosófico...",spa_Latn


In [36]:
import pandas as pd
import fasttext
import re

# Filtrar solo textos en español
df = df[df["idioma"] == "spa_Latn"].copy()

df[["text","emotion", "idioma"]]

,text,emotion,idioma
0,Acabo de ver la gran pérdida que estamos tenie...,tristeza,spa_Latn
1,¿A que vamos a reconstruir Notre Dame antes de...,tristeza,spa_Latn
2,Desde ayer andan sufriendo por el incendio y n...,ira,spa_Latn
3,Muy afectada como muchísima gente por lo ocurr...,tristeza,spa_Latn
4,Es una mierda lo que paso pero please dejen de...,disgusto,spa_Latn
...,...,...,...
8402,Dembele de mierda piernas chuecas regalaste do...,ira,spa_Latn
8404,"Dembele es un imbécil, neta es increíble lo pe...",ira,spa_Latn
8405,Puta q son desagradables los wnes del Barca en...,ira,spa_Latn
8407,"Como te odio , odio tus comentarios filosófico...",ira,spa_Latn


In [37]:
df = df.drop(columns=["idioma"] )
df['emotion'].value_counts()
# gui = show(df)

emotion
neutral     3884
alegría     1679
tristeza     979
ira          797
disgusto     153
miedo         87
Name: count, dtype: int64

In [38]:
# Eliminar solo el último punto (si existe al final)
df["text"] = df["text"].str.replace(r"\.$", "", regex=True)

In [39]:
df = df.reset_index(drop=True)
df

,text,emotion
0,Acabo de ver la gran pérdida que estamos tenie...,tristeza
1,¿A que vamos a reconstruir Notre Dame antes de...,tristeza
2,Desde ayer andan sufriendo por el incendio y n...,ira
3,Muy afectada como muchísima gente por lo ocurr...,tristeza
4,Es una mierda lo que paso pero please dejen de...,disgusto
...,...,...
7574,Dembele de mierda piernas chuecas regalaste do...,ira
7575,"Dembele es un imbécil, neta es increíble lo pe...",ira
7576,Puta q son desagradables los wnes del Barca en...,ira
7577,"Como te odio , odio tus comentarios filosófico...",ira


In [40]:
dataToken = df.copy()
# dataToken

In [41]:
import pandas as pd

# Función alternativa para contar tokens utilizando split()
def contar_tokens_simple(texto):
    return len(texto.split())

# Aplicar la función y filtrar las filas con menos de 10 tokens
dataToken['token_count'] = dataToken['text'].apply(contar_tokens_simple)
dataToken = dataToken[dataToken['token_count'] <= 3]

# Mostrar el resultado
dataToken

,text,emotion,token_count
3412,Buenas y tarde,neutral,3
5059,Emotivo discurso en,neutral,3
5776,Las Ligas ️,neutral,3
5789,levanta su título,neutral,3
5993,Mi Barcelona ganará,alegría,3
6014,"Medio tiempo, vs.",neutral,3
6253,vuelve y por,neutral,3
6405,En medio de,neutral,3


In [42]:
# Obtener los índices de las filas donde 'token_count' es 0
indices_a_eliminar = dataToken[dataToken['token_count'] <= 3].index

# Eliminar las filas del DataFrame original (df)
df = df.drop(index=indices_a_eliminar)
dataToken = dataToken.drop(index=indices_a_eliminar)


In [43]:
df = df.reset_index(drop=True)
dataToken = df.copy()
# df

In [44]:
# Aplicar la función y filtrar las filas con menos de 10 tokens
dataToken['token_count'] = dataToken['text'].apply(contar_tokens_simple)
# dataToken = dataToken[dataToken['token_count'] <= 7]

# Mostrar el resultado
dataToken

,text,emotion,token_count
0,Acabo de ver la gran pérdida que estamos tenie...,tristeza,27
1,¿A que vamos a reconstruir Notre Dame antes de...,tristeza,20
2,Desde ayer andan sufriendo por el incendio y n...,ira,16
3,Muy afectada como muchísima gente por lo ocurr...,tristeza,40
4,Es una mierda lo que paso pero please dejen de...,disgusto,13
...,...,...,...
7566,Dembele de mierda piernas chuecas regalaste do...,ira,13
7567,"Dembele es un imbécil, neta es increíble lo pe...",ira,11
7568,Puta q son desagradables los wnes del Barca en...,ira,13
7569,"Como te odio , odio tus comentarios filosófico...",ira,29


In [45]:
d = dtale.show(dataToken)
d.open_browser() 

In [46]:
df['emotion'].value_counts()

emotion
neutral     3877
alegría     1678
tristeza     979
ira          797
disgusto     153
miedo         87
Name: count, dtype: int64

In [47]:
df

,text,emotion
0,Acabo de ver la gran pérdida que estamos tenie...,tristeza
1,¿A que vamos a reconstruir Notre Dame antes de...,tristeza
2,Desde ayer andan sufriendo por el incendio y n...,ira
3,Muy afectada como muchísima gente por lo ocurr...,tristeza
4,Es una mierda lo que paso pero please dejen de...,disgusto
...,...,...
7566,Dembele de mierda piernas chuecas regalaste do...,ira
7567,"Dembele es un imbécil, neta es increíble lo pe...",ira
7568,Puta q son desagradables los wnes del Barca en...,ira
7569,"Como te odio , odio tus comentarios filosófico...",ira


In [48]:
d = dtale.show(df)
d.open_browser() 

In [ ]:
import pandas as pd
# df = pd.read_csv('../data/EMOEVAL_ES/EMOEVENT_CLEANED_v6.csv')

In [50]:
# Limpiar textos
df["text"] = df["text"].apply(limpiar_texto)

# 1. Eliminar espacio antes de coma ' ,'
df['text'] = df['text'].str.replace(r'\s+,', ',', regex=True)

# 2. Eliminar varias comas juntas ',,,,' (todas las comas consecutivas)
df['text'] = df['text'].str.replace(r',+', ',', regex=True)

# 3. Eliminar solo el último punto al final si está solo, pero no tocar los '...'
df['text'] = df['text'].str.replace(r'(?<!\.\.\.)\.$', '', regex=True)

# 4. Eliminar el espacio antes del punto ' .'
df['text'] = df['text'].str.replace(r'\s+\.', '.', regex=True)


# 5. Eliminar el espacio antes del signo de interrogación ' ?' y el signo de exclamación ' !'
df['text'] = df['text'].str.replace(r'\s+\?', '?', regex=True)
df['text'] = df['text'].str.replace(r'\s+\!', '!', regex=True)

# eliminar puntos finales
df["text"] = df["text"].str.rstrip(".")

# 6. Letras sueltas a eliminar, excluyendo conectores válidos como y, o, a, e
df['text'] = df['text'].str.replace(r'\b[b-df-hj-np-tv-z]\b', '', regex=True)

# limpiar espacios
df['text'] = df['text'].str.replace(r'\s+([;:!,?.])', r'\1', regex=True)

# Limpiar textos
df["text"] = df["text"].apply(limpiar_texto)
# Eliminar espacios extra al final otra vez por seguridad
df["text"] = df["text"].str.strip()

In [51]:
# Función para detectar emojis en un texto
emoji_pattern = re.compile("["
    u"\U0001F600-\U0001F64F"  # emoticonos
    u"\U0001F300-\U0001F5FF"  # símbolos y pictogramas
    u"\U0001F680-\U0001F6FF"  # transporte y mapas
    u"\U0001F1E0-\U0001F1FF"  # banderas (iOS)
    u"\U00002700-\U000027BF"  # Dingbats
    u"\U0001F900-\U0001F9FF"  # símbolos suplementarios
    u"\U00002600-\U000026FF"  # Miscelánea
    "]+", flags=re.UNICODE)
# Eliminar filas donde emotion es 'neutral' y el texto contiene al menos un emoji
df = df[~((df['emotion'] == 'neutral') & (df['text'].str.contains(emoji_pattern)))]

In [ ]:
df.to_csv("../data/EMOEVAL_ES/EMOEVENT_CLEANED_v10.csv", index=False)

In [ ]:
# import pandas as pd
# import dtale
# # Mostrar el DataFrame en D-Tale
# d = dtale.show(df)
# d.open_browser() 